# Facial Stress & Fatigue Detection — Kaggle Training Notebook

**CMPE 258 Final Project — Training Entry Point**

This notebook trains the full multi-task DINOv2-based model end-to-end on Kaggle. It works on either:
- **TPU v3-8** (free quota, ~30–45 min full run) — recommended
- **GPU T4 / P100** (free quota, ~60–90 min) — fallback

## Setup checklist before running
1. In Kaggle Notebook settings (right sidebar): **Accelerator → TPU VM v3-8** (or GPU if TPU unavailable)
2. **Add datasets** (right sidebar → Add Data → search):
   - `dheerajperumandla/drowsiness-dataset`
   - `msambare/fer2013`
3. **Add secret** `WANDB_API_KEY` (Add-ons → Secrets) so runs log to W&B
4. Run all cells top-to-bottom

After the run, the best model is saved to `/kaggle/working/checkpoints/best/` — download via the right sidebar.

## 1. Environment setup

In [ ]:
# Detect accelerator type
import os
import sys
import torch

USE_TPU = False
try:
    import torch_xla.core.xla_model as xm  # noqa: F401
    USE_TPU = True
    print('✓ TPU available')
except ImportError:
    print('TPU not available — falling back to GPU/CPU')

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('Torch version:', torch.__version__)

In [ ]:
# Clone our repo (replace with your actual GitHub repo URL)
REPO_URL = 'https://github.com/NMemane1/CMPE258-DeepLearning-Facial-Stress-And-Fatigue-Detection-FinalProject.git'

!cd /kaggle/working && git clone $REPO_URL repo 2>/dev/null || (cd /kaggle/working/repo && git pull)

%cd /kaggle/working/repo
!ls -la

In [ ]:
# Install required deps. mediapipe + opencv may be skipped on TPU image (we don't need face detection during training).
!pip install -q transformers>=4.40 huggingface-hub>=0.22 omegaconf scikit-learn tensorboard wandb pandas pyyaml
# Optional: pip install mediapipe opencv-python-headless  # only if you want to test the inference pipeline here

In [ ]:
# Set up W&B from Kaggle secret
from kaggle_secrets import UserSecretsClient
import os
try:
    wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
    os.environ['WANDB_API_KEY'] = wandb_key
    print('✓ W&B key loaded')
except Exception as e:
    print(f'W&B key not found ({e}); training will continue without W&B logging')
    os.environ['WANDB_DISABLED'] = 'true'

## 2. Data preparation

In [ ]:
import shutil
from pathlib import Path

DATA_ROOT = Path('/kaggle/working/data/processed')
DROWSY_DIR = DATA_ROOT / 'drowsiness'
FER_DIR = DATA_ROOT / 'fer2013'
DROWSY_DIR.mkdir(parents=True, exist_ok=True)
FER_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle mounts datasets at /kaggle/input/<slug>
# Symlink (or copy) into the layout our dataset.py expects.

def link_or_copy(src, dst):
    src = Path(src); dst = Path(dst)
    if not src.exists():
        print(f'  ! {src} not found — skipping')
        return
    if dst.exists():
        return
    try:
        os.symlink(str(src), str(dst))
    except OSError:
        shutil.copytree(str(src), str(dst)) if src.is_dir() else shutil.copy(str(src), str(dst))

# Drowsiness dataset has folder names like Closed/Open/yawn/no_yawn under a top-level
drowsy_root = Path('/kaggle/input/drowsiness-dataset')
if drowsy_root.exists():
    # Find the actual leaf dirs
    for sub in ['Closed', 'Open', 'yawn', 'no_yawn']:
        candidates = list(drowsy_root.rglob(sub))
        if candidates:
            link_or_copy(candidates[0], DROWSY_DIR / sub)

# FER-2013
fer_root = Path('/kaggle/input/fer2013')
if fer_root.exists():
    # Possible layouts:
    #   /kaggle/input/fer2013/train/<emotion>/*.png
    #   /kaggle/input/fer2013/test/<emotion>/*.png
    for split in ['train', 'test']:
        for cand in list(fer_root.rglob(split)):
            if cand.is_dir():
                link_or_copy(cand, FER_DIR / split)
                break

print('Drowsiness root contents:')
!ls -la {DROWSY_DIR} 2>/dev/null | head -20
print('\nFER root contents:')
!ls -la {FER_DIR} 2>/dev/null | head -20

In [ ]:
# Sanity-check dataset loading
sys.path.insert(0, '/kaggle/working/repo')
from src.data.dataset import DrowsinessDataset, FER2013Dataset

drowsy_ds = DrowsinessDataset(str(DROWSY_DIR))
fer_ds = FER2013Dataset(str(FER_DIR))
print(f'Drowsiness samples: {len(drowsy_ds)}')
print(f'FER-2013 samples:   {len(fer_ds)}')

## 3. Load config and customize for this run

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load('/kaggle/working/repo/src/config/config.yaml')

# Override for Kaggle environment
cfg.data.drowsiness_root = str(DROWSY_DIR)
cfg.data.fer_root = str(FER_DIR)
cfg.experiment.output_dir = '/kaggle/working/outputs'

if USE_TPU:
    cfg.training.batch_size = 64
    cfg.training.precision = 'bf16'
    cfg.data.num_workers = 2
elif torch.cuda.is_available():
    cfg.training.batch_size = 32
    cfg.training.precision = 'fp16'
    cfg.data.num_workers = 2
else:
    cfg.training.batch_size = 8
    cfg.training.precision = 'fp32'
    cfg.data.num_workers = 0

# Replace with your W&B entity name
cfg.logging.wandb_entity = 'YOUR_WANDB_USERNAME'   # ← EDIT ME

print(OmegaConf.to_yaml(cfg))

## 4. Train

In [ ]:
from src.data.transforms import build_train_transform, build_eval_transform
from src.data.dataset import (
    DrowsinessDataset, FER2013Dataset, CombinedFacialDataset,
    make_subject_grouped_split, compute_class_weights,
)
from src.models.stress_fatigue_model import StressFatigueModel
from src.training.losses import MultiTaskLoss
from src.training.trainer import Trainer
from torch.utils.data import DataLoader

# Build datasets
train_tf = build_train_transform(cfg)
eval_tf  = build_eval_transform(cfg)

drow_full = DrowsinessDataset(cfg.data.drowsiness_root, transform=None)
fer_full  = FER2013Dataset(cfg.data.fer_root, transform=None)
drow_split = make_subject_grouped_split(drow_full, cfg.data.train_split, cfg.data.val_split, cfg.data.test_split, seed=cfg.experiment.seed)
fer_split  = make_subject_grouped_split(fer_full,  cfg.data.train_split, cfg.data.val_split, cfg.data.test_split, seed=cfg.experiment.seed)

train_ds = CombinedFacialDataset([
    DrowsinessDataset(cfg.data.drowsiness_root, indices=drow_split.train, transform=train_tf),
    FER2013Dataset(cfg.data.fer_root, indices=fer_split.train, transform=train_tf),
])
val_ds = CombinedFacialDataset([
    DrowsinessDataset(cfg.data.drowsiness_root, indices=drow_split.val, transform=eval_tf),
    FER2013Dataset(cfg.data.fer_root, indices=fer_split.val, transform=eval_tf),
])
test_ds = CombinedFacialDataset([
    DrowsinessDataset(cfg.data.drowsiness_root, indices=drow_split.test, transform=eval_tf),
    FER2013Dataset(cfg.data.fer_root, indices=fer_split.test, transform=eval_tf),
])

print(f'Splits: train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=cfg.training.batch_size, shuffle=True,  drop_last=True,  num_workers=cfg.data.num_workers, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.training.batch_size, shuffle=False, drop_last=False, num_workers=cfg.data.num_workers, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.training.batch_size, shuffle=False, drop_last=False, num_workers=cfg.data.num_workers, pin_memory=True)

In [ ]:
# Class weights for balanced loss
stress_w = compute_class_weights(fer_full, 'stress', cfg.model.num_stress_classes)
fatigue_w = compute_class_weights(drow_full, 'fatigue', cfg.model.num_fatigue_classes)
print('Stress class weights: ', stress_w.tolist())
print('Fatigue class weights:', fatigue_w.tolist())

In [ ]:
# Build model + loss
model = StressFatigueModel(
    backbone_name=cfg.model.backbone,
    embedding_dim=cfg.model.embedding_dim,
    shared_hidden=cfg.model.shared_hidden,
    num_stress_classes=cfg.model.num_stress_classes,
    num_fatigue_classes=cfg.model.num_fatigue_classes,
    dropout=cfg.model.dropout,
    unfreeze_from_layer=cfg.model.unfreeze_from_layer,
)
print('Parameter summary:', model.parameter_summary())

loss_fn = MultiTaskLoss(
    stress_class_weight=stress_w,
    fatigue_class_weight=fatigue_w,
    weight_stress=cfg.training.loss_weights.stress,
    weight_fatigue=cfg.training.loss_weights.fatigue,
    weight_focal=cfg.training.loss_weights.focal if cfg.model.use_focal_loss else 0.0,
    focal_gamma=cfg.model.focal_gamma,
)

In [ ]:
# Train! (this is the main run)
trainer = Trainer(cfg, model, loss_fn, train_loader, val_loader, test_loader)
test_metrics = trainer.fit()
print('\n=== FINAL TEST METRICS ===')
import json
print(json.dumps(test_metrics, indent=2))

## 5. Evaluation plots

In [ ]:
import numpy as np
import torch
from src.evaluation.metrics import plot_confusion_matrix, plot_calibration
from src.data.dataset import STRESS_CLASS_NAMES, FATIGUE_CLASS_NAMES

model.eval()
device = next(model.parameters()).device

s_probs_list, s_tgt_list, f_probs_list, f_tgt_list = [], [], [], []
with torch.no_grad():
    for batch in test_loader:
        out = model(batch['image'].to(device))
        s_probs_list.append(torch.softmax(out.stress_logits, dim=-1).cpu().numpy())
        f_probs_list.append(torch.softmax(out.fatigue_logits, dim=-1).cpu().numpy())
        s_tgt_list.append(batch['stress_label'].numpy())
        f_tgt_list.append(batch['fatigue_label'].numpy())

s_probs = np.concatenate(s_probs_list)
f_probs = np.concatenate(f_probs_list)
s_tgt   = np.concatenate(s_tgt_list)
f_tgt   = np.concatenate(f_tgt_list)

out_dir = Path('/kaggle/working/outputs/plots')
out_dir.mkdir(parents=True, exist_ok=True)

# Confusion matrices (only over labeled samples)
s_mask = s_tgt >= 0
f_mask = f_tgt >= 0
plot_confusion_matrix(s_tgt[s_mask], s_probs[s_mask].argmax(1), STRESS_CLASS_NAMES, str(out_dir / 'cm_stress.png'))
plot_confusion_matrix(f_tgt[f_mask], f_probs[f_mask].argmax(1), FATIGUE_CLASS_NAMES, str(out_dir / 'cm_fatigue.png'))
plot_calibration(s_probs[s_mask], s_tgt[s_mask], str(out_dir / 'calibration_stress.png'))
plot_calibration(f_probs[f_mask], f_tgt[f_mask], str(out_dir / 'calibration_fatigue.png'))

print('Plots saved to', out_dir)
from IPython.display import Image as DisplayImage
DisplayImage(filename=str(out_dir / 'cm_stress.png'))

## 6. Push trained model to HuggingFace Hub

In [ ]:
# Optional: push the best checkpoint to HF Hub for the deployed Space to load.
# Requires HF_TOKEN secret in Kaggle.
try:
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token

    from huggingface_hub import HfApi, create_repo
    repo_id = 'YOUR_USERNAME/facial-stress-fatigue-dinov2'  # ← EDIT ME
    create_repo(repo_id, token=hf_token, exist_ok=True, repo_type='model')
    api = HfApi(token=hf_token)
    api.upload_folder(
        folder_path=str(Path('/kaggle/working/outputs') / cfg.experiment.name / 'checkpoints' / 'best'),
        repo_id=repo_id,
        repo_type='model',
        commit_message=f'Kaggle TPU training run — best on val={trainer.state.best_metric:.4f}',
    )
    print(f'✓ Uploaded to https://huggingface.co/{repo_id}')
except Exception as e:
    print(f'HF upload skipped: {e}')
    print('Download the checkpoint manually from /kaggle/working/outputs')

In [ ]:
# Zip outputs for easy download
!cd /kaggle/working && zip -r outputs.zip outputs/ 2>&1 | tail -5
!ls -lh /kaggle/working/outputs.zip